<a href="https://colab.research.google.com/github/andrea-t94/airflow-net/blob/master/research/finetuning/notebooks/01_finetune_no_unsloth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finetuning Qwen2.5 on Airflow DAGs (without Unsloth, with Flash Attention 2)

This notebook demonstrates how to fine-tune the **Qwen/Qwen2.5-Coder-1.5B-Instruct** model on a dataset of **Airflow DAGs** using standard HuggingFace libraries (`transformers`, `peft`, `trl`) — without relying on Unsloth.

It uses **Flash Attention 2** for faster and more memory-efficient attention computation. This requires an **Ampere or newer GPU** (e.g., A100, A10G, L4).

### Please note
**1. This notebook has been developed and tested on Google Colab with an A100 GPU**, which is the recommended environment for reproduction. Running it in other environments may require modifications to the setup and installation steps.


**2. If you want to fine-tune the model with your own settings**, you will need to modify:
- **`NEW_MODEL_NAME`** in the configuration cell — change the username to your own Hugging Face account (e.g., `your-username/qwen2.5-1.5b-airflow-instruct`)
- **Hugging Face Token** — ensure you have write permissions to push models to your account

## 1. Setup & Installation
We install the necessary libraries for fine-tuning with QLoRA.

In [ ]:
%%capture
import os
import torch

!pip install transformers accelerate peft trl bitsandbytes datasets huggingface_hub

# Install Flash Attention 2 (requires Ampere+ GPU)
!pip install flash-attn --no-build-isolation

In [ ]:
import flash_attn

# Verify GPU and PyTorch
print(f"PyTorch version: {torch.__version__}")
print(f"Flash Attention version: {flash_attn.__version__}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU Detected: {gpu_name}")
    major_version, minor_version = torch.cuda.get_device_capability()
    if major_version >= 8:
        print("\u2705 GPU supports bfloat16 and Flash Attention 2 (Ampere or newer).")
    else:
        raise RuntimeError(f"This notebook requires an Ampere+ GPU (sm_80+), but found sm_{major_version}{minor_version}. "
                           "Use the SDPA notebook (01_finetune_no_unsloth_sdpa.ipynb) instead.")
else:
    raise RuntimeError("No GPU detected! Please change runtime type to GPU in 'Runtime > Change runtime type'.")

## 2. Configuration & Authentication
Log in to Hugging Face to access datasets and push your model.

In [ ]:
from huggingface_hub import login

# Try to get token from Colab secrets, otherwise prompt
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=True)
except:
    print("Please provide your Hugging Face Token (Permissions: Write)")
    login(add_to_git_credential=True)

In [ ]:
# Project Configuration
BASE_MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
DATASET_NAME = "andrea-t94/airflow-dag-dataset"

# Output Model Name (Change username if needed)
NEW_MODEL_NAME = "andrea-t94/qwen2.5-1.5b-airflow-instruct"

# Training Parameters
MAX_SEQ_LENGTH = 4096 # Fits most DAG files
LOAD_IN_4BIT = True   # Enable 4-bit quantization (QLoRA) to save memory

## 3. Load Model with QLoRA + Flash Attention 2
We load the model in 4-bit precision using `bitsandbytes` and apply LoRA adapters via `peft`. Flash Attention 2 is enabled for faster attention computation.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# A100 supports bfloat16
compute_dtype = torch.bfloat16

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
tokenizer.padding_side = "right"

# Load model with Flash Attention 2
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="flash_attention_2",
    torch_dtype=compute_dtype,
)

print(f"\u2705 Model loaded with Flash Attention 2")

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# LoRA config (same as the Unsloth notebook)
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Load & Format Dataset
We specificy a formatting function to apply the ChatML template (which Qwen uses) to our dataset.

In [ ]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset(DATASET_NAME)

# Inspect dataset sizes
print(f"Train size: {len(dataset['train'])}")
if 'eval' in dataset: print(f"Eval size:  {len(dataset['eval'])}")

# Format function for ChatML
# The dataset should have a 'messages' column matching standard chat format
def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        # Apply chat template but do NOT tokenize yet
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

# Apply formatting
dataset = dataset.map(formatting_prompts_func, batched=True)

## 5. Training
Configure the `SFTTrainer`. We use `gradient_accumulation_steps` to simulate a larger batch size.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset.get("eval"),
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = MAX_SEQ_LENGTH,
        dataset_num_proc = 2,
        packing = True, # Set to True to speed up training if sequence len is variable and usually are shorter than max_seq_len
        per_device_train_batch_size = 4,  # Increase if GPU memory allows
        gradient_accumulation_steps = 8,   # effective_batch = per_device_train_batch_size*gradient_accumulation_steps
        warmup_steps = 10,
        max_steps = -1,                   # Set to -1 for full epochs
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = False,
        bf16 = True,
        logging_steps = 1,
        optim = "adamw_8bit",             # Use 8-bit optimizer to save memory
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        save_strategy = "steps",
        eval_strategy = "steps",
        eval_steps = 100,
        gradient_checkpointing = True,
        gradient_checkpointing_kwargs = {"use_reentrant": False},
    ),
)

In [ ]:
# Start Training
trainer_stats = trainer.train()

## 6. Save & Push to Hub
We save LoRA adapters and the full merged model, then push to Hugging Face Hub.

In [ ]:
# 1. Save LoRA Adapters only (Small file size, fast)
model.save_pretrained("lora_adapters")
tokenizer.save_pretrained("lora_adapters")
model.push_to_hub(f"{NEW_MODEL_NAME}-lora", token=True)
tokenizer.push_to_hub(f"{NEW_MODEL_NAME}-lora", token=True)

In [ ]:
# 2. Save Merged Model (Full model for direct inference)
from peft import AutoPeftModelForCausalLM

print("Merging LoRA weights into base model...")
merged_model = model.merge_and_unload()

# Save locally
merged_model.save_pretrained("merged_model", safe_serialization=True)
tokenizer.save_pretrained("merged_model")

# Push to Hub
print("Pushing merged model to Hub...")
merged_model.push_to_hub(NEW_MODEL_NAME, token=True, safe_serialization=True)
tokenizer.push_to_hub(NEW_MODEL_NAME, token=True)
print(f"Saved merged model to https://huggingface.co/{NEW_MODEL_NAME}")

In [ ]:
# 3. Convert to GGUF (for Ollama/Llama.cpp)
# Without Unsloth, we use llama.cpp's convert script directly
print("Converting to GGUF...")

# Install llama.cpp
!git clone https://github.com/ggerganov/llama.cpp.git /tmp/llama_cpp 2>/dev/null || true
!cd /tmp/llama_cpp && pip install -r requirements.txt 2>/dev/null

# Convert to GGUF f16 first
!python /tmp/llama_cpp/convert_hf_to_gguf.py merged_model --outfile merged_model.f16.gguf --outtype f16

# Quantize to Q4_K_M
!cd /tmp/llama_cpp && make -j quantize 2>/dev/null
!/tmp/llama_cpp/llama-quantize merged_model.f16.gguf merged_model.Q4_K_M.gguf Q4_K_M

# Upload GGUF to Hub
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj="merged_model.Q4_K_M.gguf",
    path_in_repo="qwen2.5-coder-1.5b-instruct.Q4_K_M.gguf",
    repo_id=NEW_MODEL_NAME,
    token=True
)
print(f"GGUF uploaded to https://huggingface.co/{NEW_MODEL_NAME}")